In [1]:
import copy
import correctionlib
import numpy as np
import pandas as pd
import awkward as ak
from pathlib import Path
from coffea.util import load


def get_os_fr(lepton_pt, lepton_eta, kind, year):
    """
    Get the fake rate for a lepton passing the opposite-sign (OS) fake rate measurement

    The fake rate f(pT, η) represents the probability that a nonprompt lepton
    (originating from a jet, heavy-flavor decay, or photon conversion)
    is misidentified as a *tight* lepton, given that it passes a *loose* selection.
    """
    # Define detector regions and pT thresholds for each lepton kind
    # Determine if the lepton is in the barrel or endcap region of the detector
    barrel_cut = {"electron": 1.479, "muon": 1.2}
    pt_cut = {"electron": 7.0, "muon": 5.0}

    lepton_pt = ak.Array(lepton_pt)
    lepton_eta = ak.Array(lepton_eta)
    is_barrel = lepton_eta < barrel_cut[kind]
    lepton_in_binning = (lepton_pt >= pt_cut[kind]) & (lepton_pt <= 80.0)

    # Mask values to ensure consistent binning
    lepton_in_pt = ak.fill_none(lepton_pt.mask[lepton_in_binning], 10.0)
    in_is_barrel = ak.fill_none(is_barrel.mask[lepton_in_binning], True)

    # Load fake-rate correction set from JSON file
    json_path = f"../analysis/data/{year}_os_fr_weight.json.gz"
    cset = correctionlib.CorrectionSet.from_file(str(json_path))
    
    # Evaluate the fake rate from the correction set
    lepton_fr = cset["os_fake_rate"].evaluate(
        lepton_in_pt, ak.values_astype(in_is_barrel, float), kind
    )

    # Set FR = 0 for leptons outside of the defined measurement phase space
    fake_rates = ak.fill_none(ak.where(lepton_in_binning, lepton_fr, ak.zeros_like(lepton_fr)), 0)
    
    return fake_rates 



def add_os_fake_rates(df, region, year):
    """
    Compute and append opposite-sign fake rates for leptons l3 and l4 in a given control region (1FCR or 2FCR)

    - 1FCR ("one fake control region") corresponds to events with 3 tight + 1 fail leptons (3P1F).
    - 2FCR ("two fake control region") corresponds to 2 tight + 2 fail leptons (2P2F).
    - The FRs are applied to the failing leptons (l3 and l4) to predict their contribution to the signal region (SR) 
    """
    for lep in ["l3", "l4"]:
        pt_col = f"{lep}_pt_{region}"
        eta_col = f"{lep}_eta_{region}"
        pdgid_col = f"{lep}_pdgid_{region}"
        out_col = f"{lep}_os_fr_{region}"

        # Build mask to select valid entries 
        valid_entries = df[pt_col].notna() & df[eta_col].notna() & df[pdgid_col].notna()
        df[out_col] = np.nan

        pt = df.loc[valid_entries, pt_col].to_numpy()
        eta = df.loc[valid_entries, eta_col].to_numpy()
        pdgid = df.loc[valid_entries, pdgid_col].to_numpy()

        # Identify electrons and muons
        is_elec = np.abs(pdgid) == 11
        is_muon = np.abs(pdgid) == 13

        # Retrieve fake rates for each lepton kind
        fr_values = np.zeros_like(pt)
        if np.any(is_elec):
            fr_values[is_elec] = ak.to_numpy(
                get_os_fr(pt[is_elec], eta[is_elec], kind="electron", year=year)
            )
        if np.any(is_muon):
            fr_values[is_muon] = ak.to_numpy(
                get_os_fr(pt[is_muon], eta[is_muon], kind="muon", year=year)
            )
            
        # Store FR values in the dataframe
        df.loc[valid_entries, out_col] = fr_values

    return df

The expected reducible background in the signal region is given by

$$
N^{bkg}_{SR} = \left(1 - \frac{N^{ZZ}_{3P1F}}{N_{3P1F}}\right) 
\sum_{j}^{N_{3P1F}} \frac{f_a^j}{1 - f_a^j}
- \sum_{i}^{N_{2P2F}} \frac{f_3^i}{1 - f_3^i} \frac{f_4^i}{1 - f_4^i}
$$

- The first term extrapolates from the 3P1F control region to the signal region, subtracting the contamination from prompt ZZ events. 
- The second term subtracts double-counted contributions from events with two nonprompt leptons (2P2F region).

In [2]:
years = ["2022preEE", "2022postEE", "2023preBPix", "2023postBPix"]
channels = ["4mu", "4e", "2e2mu", "2mu2e"]

zplusx_estimate = {}
for year in years:
    zplusx_estimate[year] = {}
    for channel in channels:
        # Compute number of 3P1F Data events
        data_histograms = load(f'../outputs/zplusll_os/{year}/Data.coffea')["Data"]
        data_zll_mass_1fcr = data_histograms[f'zll_mass_{channel}_1fcr'][{"variation":"nominal"}]
        n_data_3p1f = data_zll_mass_1fcr.values().sum()
    
        # Compute number of 3P1F ZZ events
        zz_histograms = load(f'../outputs/zplusll_os/{year}/Diboson.coffea')["Diboson"]
        zz_zll_mass_1fcr = zz_histograms[f"zll_mass_{channel}_1fcr"][{"variation":"nominal"}]
        n_zz_3p1f = zz_zll_mass_1fcr.values().sum()
        
        # Load Data parquet file into a pandas dataframe and add per-lepton OS fake rates
        data_df = pd.read_parquet(f'../outputs/zplusll_os/{year}/Data.parquet')
        for region in ["1fcr", "2fcr"]:
            data_df = add_os_fake_rates(data_df, region, year=year)
    
        # Compute sum of 2P2F weights
        # Each event contributes with product of fake factors for the two failing leptons.
        # Weight: (f3/(1-f3))*(f4/(1-f4))
        channel_2fcr_events = data_df[f"zll_mass_{channel}_2fcr"].notna()
        weights_2p2f = (data_df.l3_os_fr_2fcr / (1 - data_df.l3_os_fr_2fcr)) * (data_df.l4_os_fr_2fcr / (1 - data_df.l4_os_fr_2fcr))
        sum_weights_2p2f = weights_2p2f[channel_2fcr_events].sum()
    
        # compute sum of 3P1F weights 
        # Each 3P1F event contributes with the fake factor f/(1-f) for the failing lepton.
        data_df["weights_3p1f"] = np.nan
        # Case 1: l3 is tight (failing lepton is l4)
        mask_l3 = data_df.l3_istight_1fcr == True
        data_df.loc[mask_l3, "weights_3p1f"] = (
            data_df.loc[mask_l3, "l3_os_fr_1fcr"] / (1 - data_df.loc[mask_l3, "l3_os_fr_1fcr"])
        )
        # Case 2: l4 is tight (failing lepton is l3)
        mask_l4 = data_df.l4_istight_1fcr == True
        data_df.loc[mask_l4, "weights_3p1f"] = (
            data_df.loc[mask_l4, "l4_os_fr_1fcr"] / (1 - data_df.loc[mask_l4, "l4_os_fr_1fcr"])
        )
        # Sum only over events in the current channel and region
        channel_2fcr_events = data_df[f"zll_mass_{channel}_1fcr"].notna()
        sum_weights_3p1f = data_df[channel_2fcr_events].weights_3p1f.sum()
    
        # Compute estimated number of Z+X events in signal region
        n_sr_bkg = (1 - n_zz_3p1f/n_data_3p1f) * sum_weights_3p1f - sum_weights_2p2f

        zplusx_estimate[year][channel] = n_sr_bkg


pd.DataFrame(zplusx_estimate).T

,4mu,4e,2e2mu,2mu2e
2022preEE,6.491424,3.337846,3.197386,3.847793
2022postEE,13.224687,10.198200,14.943629,11.588095
2023preBPix,11.787283,7.156391,6.639530,9.724362
2023postBPix,5.109555,3.158902,5.107512,5.708860
